# 🍽️ **Modelado de Series de Tiempo con SARIMAX - Caso: Visitantes a Restaurantes**


En este notebook se realiza un análisis explicativo del modelo **SARIMAX** aplicado a una serie de tiempo real que representa el número total de visitantes diarios a cuatro restaurantes en Japón desde enero de 2016 hasta abril de 2017.

Utilizaremos como variable objetivo la columna **`total`**, que representa el número total de visitantes. Como variable exógena, se incluirá la columna **`holiday`**, que indica si un día es feriado (`1`) o no (`0`).


## 🧠 **¿Qué es el modelo SARIMAX?**


El modelo **SARIMAX** (Seasonal AutoRegressive Integrated Moving Average with eXogenous regressors) extiende el modelo ARIMA incorporando tanto:

- Componente estacional (SARIMA)
- Variables exógenas (X)

La forma general del modelo es:

$$SARIMAX(p,d,q)(P,D,Q,s) + X_t$$

donde:

- $p, d, q$: órdenes del modelo ARIMA
- $P, D, Q$: órdenes del componente estacional
- $s$: periodicidad estacional (e.g. semanal: $s=7$)
- $X_t$: variable(s) exógena(s) en tiempo $t$


## 📥 **Carga y preparación de los datos**

In [ ]:
import pandas as pd

df = pd.read_csv("RestaurantVisitors.csv")
df["date"] = pd.to_datetime(df["date"])
df.set_index("date", inplace=True)
df = df.sort_index()
df[["total", "holiday"]].head()

In [ ]:
df.tail(50)

## 📈 **Visualización de la serie de visitantes totales**

In [ ]:
import matplotlib.pyplot as plt

df["total"].plot(title="Visitas totales a restaurantes (2016-2017)", figsize=(12, 4))
plt.grid()
plt.ylabel("Visitantes")
plt.show()

## 🧪 **Exploración de la variable exógena: feriados**

In [ ]:
df[["total", "holiday"]].plot(subplots=True, figsize=(10, 5), title=["Total de visitantes", "Feriado (1=sí)"])
plt.grid()
plt.show()

## ⚙️ **Descomposición de la serie**

Descompondremos la serie para observar su estructura en tendencia, estacionalidad y residuales.

In [ ]:
df_data = df[~df["total"].isna()]

In [ ]:
from statsmodels.tsa.seasonal import seasonal_decompose

resultado = seasonal_decompose(df_data["total"], model="additive", period=7)
resultado.plot()
plt.show()

## 🧪 **Prueba de estacionariedad: Dickey-Fuller**


Aplicamos la prueba ADF para verificar si la serie es estacionaria. Una serie estacionaria es aquella cuyas propiedades estadísticas (media, varianza) no cambian en el tiempo.


In [ ]:
from statsmodels.tsa.stattools import adfuller

resultado_adf = adfuller(df_data["total"])
print(f"Valor p: {resultado_adf[1]}")

## 🔁 **Diferenciación de la serie $(d=1)$**

In [ ]:
df_data["diff"] = df_data["total"].diff()
df_data["diff"].dropna().plot(title="Primera diferenciación", figsize=(10, 4))
plt.grid()
plt.show()

adf_diff = adfuller(df_data["diff"].dropna())
print(f"Valor p tras diferenciación: {adf_diff[1]}")

## 📉 **ACF y PACF para identificación de parámetros**

Estas funciones permiten estimar los órdenes $p$ y $q$.

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plot_acf(df_data["diff"].dropna(), lags=30)
plt.show()

plot_pacf(df_data["diff"].dropna(), lags=30)
plt.show()

## 🔮 **Ajuste del modelo SARIMAX**


Usamos como variable objetivo `total` y como variable exógena `holiday`. Se eligen parámetros tentativos basados en ACF, PACF y la naturaleza semanal de los datos (estacionalidad de 7 días).

$$SARIMAX(1,1,1)(1,0,1,7)$$


In [ ]:
from statsmodels.tsa.statespace.sarimax import SARIMAX

modelo = SARIMAX(df_data["total"], exog=df_data[["holiday"]], order=(1,1,1), seasonal_order=(1,0,1,7))
resultado = modelo.fit()
print(resultado.summary())

## 📊 **Evaluación del modelo: residuos**

In [ ]:
import numpy as np

residuos = np.abs(resultado.resid)
residuos.plot(title="Residuos del modelo SARIMAX", figsize=(10, 4))
plt.grid()
plt.show()

## 📉 **Autocorrelación de residuos**

In [ ]:
from statsmodels.graphics.tsaplots import plot_acf

plot_acf(residuos.dropna(), lags=30)
plt.title("ACF de los residuos")
plt.show()

## 📦 **Predicción fuera de muestra**

Se realiza predicción de los últimos 30 días utilizando la variable exógena `holiday`.

In [ ]:
len(df_data)-1 + 40

In [ ]:
df.tail(60)

In [ ]:
n_dias = 30
pred = resultado.get_prediction(
    start=len(df_data)-1,
    end=len(df_data)-1 + 40,
    exog=df[["holiday"]].iloc[-40: ],
    dynamic=False
)

ax = pred.predicted_mean.plot(label="Predicción", figsize=(12, 4))
ax.axvline(x='2017-05-05', color='red', alpha = 1);
df_data["total"].iloc[-60:].plot(label="Real")
plt.legend()
plt.title("Predicción vs Real (últimos 60 días)")
plt.grid()
plt.show()

In [ ]:
pred.predicted_mean

## ✅ **Conclusión**


El modelo SARIMAX permite incorporar eventos externos (como feriados) que afectan la dinámica de la serie de tiempo. En este caso, los feriados impactan la demanda de visitantes a restaurantes, y su incorporación mejora la capacidad predictiva del modelo respecto a un ARIMA tradicional.

Este enfoque es útil para series con factores externos conocidos que influyen en el comportamiento futuro.
